In [ ]:
import glob
import os
from functools import reduce

import ipywidgets
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns


In [ ]:
PATH_DATA = os.path.join("..", "data")

PATH_RETURNLVLS = os.path.join(
    PATH_DATA, "models", "mev_nn", "final_ensemble", "results"
)

In [ ]:
## Select the file for the desired return level
FILES = glob.glob(os.path.join(PATH_RETURNLVLS, "*.csv"))
FILE_NAMES = sorted(
    [os.path.basename(x) for x in glob.glob(os.path.join(PATH_RETURNLVLS, "*.csv"))]
)

csv_files = ipywidgets.SelectMultiple(
    options=FILE_NAMES,
    value=FILE_NAMES,
    description="Choose return period:",
    disabled=False,
)
csv_files

In [ ]:
df_temp = pd.read_csv(os.path.join(PATH_RETURNLVLS, csv_files.value[0]))

avail_cols = sorted(list(set(df_temp.columns) - set(["lon", "lat"])))

sel_cols = ipywidgets.SelectMultiple(
    options=avail_cols, value=avail_cols, description="Choose columns:", disabled=False
)

sel_cols

In [ ]:
df_append = []

# append all files together
for file in csv_files.value:
    df_temp = pd.read_csv(os.path.join(PATH_RETURNLVLS, file))
    df_temp = df_temp[["lon", "lat"] + list(sel_cols.value)]

    for col in sel_cols.value:
        df_temp[col]
        df_temp.rename(
            columns={col: f"{os.path.splitext(file)[0]}_{col}"}, inplace=True
        )

    df_append.append(df_temp)

In [ ]:
return_data = reduce(
    lambda x, y: pd.merge(x, y, on=["lon", "lat"], how="outer"), df_append
)

In [ ]:
return_data

In [ ]:
for boxenplot in [False, True]:
    filename = f"return_levels_trend{'_boxenplot' if boxenplot else '_boxplot'}"

    for var in sel_cols.value:
        is_conf_band = "conf" in var

        # Prepare data for boxplot
        plot_data = return_data[[colname for colname in return_data.columns if colname.endswith(var)]].copy() / 10

        # Rename columns for display
        plot_data.columns = ['10 years', '20 years', '30 years']

        # Reshape data for seaborn boxplot
        plot_data_melted = plot_data.melt(var_name='Period', value_name='Median Value')

        # Create boxplot
        plt.rcParams.update({'font.size': 36})
        ax = plt.figure(figsize=(15, 10))

        kwargs = {
            "data": plot_data_melted,
            "x": 'Period',
            "y": 'Median Value'
        }

        if boxenplot:
            sns.boxenplot(**kwargs)
        else:
            sns.boxplot(**kwargs)

        if not is_conf_band:
            plt.axhline(y=5, linestyle='--', color='lightgreen', alpha=0.7)

        title = 'Confidence Interval' if is_conf_band else 'Return levels'
        title = '98% ' + title if '98p' in var else title

        plt.title(title)
        plt.ylabel('[cm]' if is_conf_band else 'hailstone size [cm]')
        plt.xlabel('period')
        plt.tight_layout()
        plt.savefig(os.path.join(PATH_RETURNLVLS, f"{filename}_{var}.png"), bbox_inches="tight")
        plt.savefig(os.path.join(PATH_RETURNLVLS, f"{filename}_{var}.pdf"), bbox_inches="tight")

        plt.close()